# 02 — Train a DQN Model Offline

Load the dataset collected by `01_collect_dataset.ipynb` and train a MOUSE model entirely from stored data — no live environment needed.

A MOUSE model is assembled from three independent pieces:

| Component | Role |
|---|---|
| `StepEmbedder` | Converts each `(action, observation, reward, done)` step into a token sequence the backbone can process |
| `Qwen3Backbone` | A transformer that reads those tokens and builds up context over the sequence |
| `DiscreteActionValueHead` | Maps the backbone's output to one Q-value per action |

`Qwen3Backbone` downloads `Qwen/Qwen3-0.6B` from the Hub on first run and keeps only the first `num_layers` transformer layers. Its `hidden_dim` (1024 for this checkpoint) is the single number that connects all three pieces — no manual dimension matching required.

In [1]:
import torch

from mouse_core.data import DataLoader, load_stores_from_hub
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead

# Hub dataset repo written by 01_collect_dataset.ipynb.
DATASET_ID = "mouse-example-dataset"

# Hub model repo where the trained checkpoint is uploaded.
MODEL_ID = "mouse-example-model"

# FrozenLake has four discrete actions, matching the DQN head width.
MAX_ACTIONS = 4

# All slots use 4×4 maps (16 discrete states, 0–15).
MAX_OBS_DISCRETE = 16

TRAIN_STEPS = 4000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load data

`load_stores_from_hub` downloads the named dataset repo and reconstructs the four `Datastore` objects — one per maze layout — that were uploaded by notebook 01.

In [2]:
stores = load_stores_from_hub(DATASET_ID, split="train")
for s in stores:
    print(s)

Datastore(name='proc_frozenlake_0', steps=1500)
Datastore(name='proc_frozenlake_1', steps=1500)
Datastore(name='proc_frozenlake_2', steps=1500)
Datastore(name='proc_frozenlake_3', steps=1500)


## DataLoader

`DataLoader` slices one or more `Datastore`s into fixed-length sequence batches and mixes them automatically — no manual merging required.

- **`sequence_length=16`** — each item in a batch is 16 consecutive steps from a single slot.
- **`batch_size=32`** — 32 sequences per call to `next_batch()`.
- **`prefetch=4`** — pre-loads 4 batches in the background to keep the GPU fed.

`next_batch()` returns a `list[list[dict]]` of shape `[batch][sequence]` — one inner list per sequence, one dict per step. Pass it directly to `model.forward`.

In [3]:
loader = DataLoader(stores, sequence_length=16, batch_size=32, prefetch=4, num_workers=0)

## Build the model

All three components are sized by `backbone.hidden_dim`, so they connect without any manual dimension matching.

**`StepEmbedder`** — each entry in `modalities` describes one field from the step dict and how to embed it:
- `"discrete"` — looks up a learned vector from a small vocabulary. Use for integer-valued fields like actions, observations, and done codes.
- `"rff"` — embeds a continuous scalar using random Fourier features. Use for rewards or any float value.
- `"learnable"` — a free learnable token with no input, giving the model scratch space it can write to without consuming an observation slot.

**`DiscreteActionValueHead`** — a small MLP that predicts one Q-value per action. `scale=0.01` initialises output weights near zero so Q-values start small and stable.

**`Model`** — wraps encoder, backbone, and head(s) into a single object with a unified `forward` call.

In [4]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B", num_layers=2)

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {"name": "action",      "embed": "discrete",  "vocab_size": MAX_ACTIONS,      "std": 0.02, "tokens": 1},
        {"name": "observation", "embed": "discrete",  "vocab_size": MAX_OBS_DISCRETE, "std": 0.02, "tokens": 1},
        {"name": "reward",      "embed": "rff",       "std": 0.02, "in_min": 0.01, "in_max": 1.0, "tokens": 1},
        {"name": "done",        "embed": "discrete",  "vocab_size": 5,               "std": 0.02, "tokens": 1},
        {"name": "scratch",     "embed": "learnable", "tokens": 1},
    ],
    concat_modalities=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.01,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(16, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
      (scratch): LearnableEmbedder()
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-1): 2 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linea

## Train

Each iteration:
1. Sample a batch of transition sequences from the data loader.
2. Run both the online network and a frozen target network to get Q-value predictions.
3. Compute the Bellman loss — `objective(objective_data, predictions)` returns `(loss, metrics)`.
4. Back-propagate, clip gradients, and step the optimizer.
5. Slowly copy online weights → target network (Polyak/EMA update) for stable TD targets.

### Discount factors and done codes

MOUSE uses five `done` codes. `DqnObjective` maps each to its own bootstrap discount:

| done | Meaning | Discount |
|---|---|---|
| 0 | Running | `gamma` |
| 1 | Episode done (not last in task) | `gamma_episode_terminal` |
| 2 | Episode truncated (not last in task) | `gamma_episode_truncated` |
| 3 | Task done (last episode) | `gamma_task_terminal` |
| 4 | Task truncated (last episode) | `gamma_task_truncated` |

Episodes within a task share context (same map, same goal), so we bootstrap across episode boundaries (`gamma_episode_* = 1.0`). At a task boundary the context resets entirely and the next task's Q-values carry no useful signal, so we suppress bootstrapping there (`gamma_task_* = 0.0`).

In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
objective = DqnObjective(
    gamma=1.0,
    gamma_episode_terminal=1.0,   # bootstrap across episode boundaries within a task
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,      # no bootstrap across task boundaries
    gamma_task_truncated=0.0,
    tau=0.005,
)

for step in range(TRAIN_STEPS):

    batch = loader.next_batch()

    predictions, objective_data, _ = model(batch)
    loss, _ = objective(objective_data, predictions)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.polyak_update(action_value_tau=objective.tau)

    if step % 100 == 0:
        print(f"step={step}  loss={loss}")

loader.close()
print("Training finished.")

step=0  loss=0.09681043773889542
step=100  loss=0.009312343783676624
step=200  loss=0.0036753893364220858
step=300  loss=0.005776472855359316
step=400  loss=0.006661003455519676
step=500  loss=0.0058859181590378284
step=600  loss=0.009079876355826855
step=700  loss=0.010435076430439949
step=800  loss=0.01088035386055708
step=900  loss=0.008601727895438671
step=1000  loss=0.008268418721854687
step=1100  loss=0.020117362961173058
step=1200  loss=0.019671225920319557
step=1300  loss=0.014427075162529945
step=1400  loss=0.024930506944656372
step=1500  loss=0.045049358159303665
step=1600  loss=0.029818760231137276
step=1700  loss=0.02283382974565029
step=1800  loss=0.030522767454385757
step=1900  loss=0.022083356976509094
step=2000  loss=0.02572084777057171
step=2100  loss=0.031726088374853134
step=2200  loss=0.05256630852818489
step=2300  loss=0.03888045623898506
step=2400  loss=0.059707626700401306
step=2500  loss=0.06719594448804855
step=2600  loss=0.07996546477079391
step=2700  loss=0.0

## Push to the Hub

Save the trained model to your Hugging Face account. Notebook 03 loads it from the same `MODEL_ID`.

In [6]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")

Pushed to https://huggingface.co/micahr234/mouse-example-model
